In [1]:
!rm ./data/opus/opus_matched.json

rm: ./data/opus/opus_matched.json: No such file or directory


In [3]:
!python step1_cornell_and_imdb.py     # ~2 min (downloads 160 MB)
!python step2_parallel_subtitles.py   # ~20-40 min (downloads 1.7 GB + range requests)
!python step3_build_dataset.py        # ~30 sec (merges cached data)

STEP 1a: Download Cornell Movie-Dialogs Corpus
  Downloading: http://www.cs.cornell.edu/~cristian/data/cornell_movie_dialogs_corpus.zip
  9/9 MB (100%)
  Extracting...

STEP 1b: Parse Cornell data
  617 movies
  9035 characters
  304446 dialogue lines
  83097 conversations
  Saved parsed data to ./data/cornell/parsed_cornell.json

STEP 1c: Resolve IMDB IDs
  Downloading: https://datasets.imdbws.com/title.basics.tsv.gz
  209/209 MB (100%)
  Building title lookup (~30 seconds)...
  973842 IMDb entries loaded
  Matched 596/617 movies → saved to ./data/cornell/imdb_mapping.json

STEP 1 COMPLETE
  Movies:        617
  Characters:    9035
  Lines:         304446
  Conversations: 83097
  IMDB matched:  596

  Files created:
    ./data/cornell/parsed_cornell.json
    ./data/cornell/imdb_mapping.json
Looking for 596 movies from Cornell

STEP 2a: Download alignment XML (119 MB)
  Downloading: https://object.pouta.csc.fi/OPUS-OpenSubtitles/v2018/xml/en-zh_cn.xml.gz
  119/119 MB (100%)

STEP 2b: P

In [4]:
from lookup_movie import *

In [5]:
movie_metadata_df = list_all_movies()

In [51]:
movie_metadata_df.title.unique()

array(['casino', "it's a wonderful life", 'titanic', 'magnolia',
       'his girl friday', 'raging bull', 'sideways', 'meet joe black',
       'goodfellas', 'grand hotel', "all the president's men",
       'apocalypse now', 'watchmen', 'the talented mr. ripley',
       'fight club', 'all about eve', 'what women want',
       'good will hunting', 'malcolm x', 'reindeer games', 'confidence',
       'jerry maguire', 'pearl harbor',
       'eternal sunshine of the spotless mind', 'enemy of the state',
       'jackie brown', 'ninotchka', 'ed wood', 'some like it hot',
       'wag the dog', 'the deer hunter', 'the big lebowski',
       'fear and loathing in las vegas', 'dog day afternoon',
       'hotel rwanda', 'thirteen days', 'wall street',
       "there's something about mary", 'pretty woman', 'get shorty',
       'bringing out the dead', 'thelma & louise', 'the negotiator',
       'l.a. confidential', 'mr. deeds goes to town', 'the rock',
       'as good as it gets', 'contact', 'the bel

In [89]:
search_movie("fight club")

,cornell_id,imdb_id,title,year,genres,characters,conversations,parallel_pairs
0,m348,0137523,fight club,1999,"['crime', 'drama', 'mystery', 'thriller']",17,188,2070


In [90]:
convo_df_script = get_conversations("fight club")

fight club (1999): 188 conversations, 779 turns


In [91]:
convo_df_script.speaker.unique()

array(['ANGEL FACE', 'JACK', 'BANDAGED PROPRIETOR', 'BOB', 'BOSS',
       'DETECTIVE STERN', 'INTERN', 'MARLA', "TYLER'S VOICE",
       'WOUNDED BARTENDER', 'SECURITY TFM', 'LEADER', 'TYLER', 'VOICE',
       'RICKY', 'LOU', 'RAYMOND'], dtype=object)

In [95]:
convo_df_script[(convo_df_script.speaker == "TYLER")&
            ((convo_df_script.participant_1 == "JACK")|
            (convo_df_script.participant_2 == "TYLER"))]

,conversation_id,participant_1,participant_2,turn_index,line_id,speaker,speaker_id,gender,credit_pos,text
327,78,JACK,TYLER,0,L212212,TYLER,u5269,m,2,One minute. This is the beginning. We're at ...
329,79,JACK,TYLER,0,L212217,TYLER,u5269,m,2,It's getting exciting now.
332,80,JACK,TYLER,1,L212220,TYLER,u5269,m,2,Look what we've accomplised. Thirty seconds.
334,81,JACK,TYLER,0,L212402,TYLER,u5269,m,2,"Two, equal parts gasoline and diet cola. Thre..."
337,82,JACK,TYLER,1,L212405,TYLER,u5269,m,2,Tyler Durden.
...,...,...,...,...,...,...,...,...,...,...
770,185,RAYMOND,TYLER,1,L212975,TYLER,u5269,m,2,Why?
772,185,RAYMOND,TYLER,3,L212977,TYLER,u5269,m,2,"What did you want to be, Raymond K. Hessel?"
773,186,RAYMOND,TYLER,0,L212981,TYLER,u5269,m,2,Animals.
775,186,RAYMOND,TYLER,2,L212983,TYLER,u5269,m,2,Stuff. That means you have to get more school...


In [98]:
convo_df_zh_en = get_parallel_df("fight club")

fight club (1999): 2070 aligned EN↔ZH pairs


In [123]:
test_str = r"\bhessel\b"

# 1. Save the filtered dataframe
filtered_df = convo_df_zh_en[
    convo_df_zh_en["en"].str.contains(test_str, case=False, na=False, regex=True)
]

# 2. Check if it's empty before trying to print
if not filtered_df.empty:
    print(filtered_df.iloc[0]['en'], "\n", filtered_df.iloc[0]['zh'])
else:
    print("No matches found for:", test_str)

Raymond K Hessel . 1320 SE Banning , Apartment A. 
 雷蒙凯艾索1320贝宁街A室


In [103]:
convo_df_zh_en

,line_number,en,zh
0,1,People always ask me if I know Tyler Durden .,人们总是问我认不认识泰勒德顿
1,2,Three minutes .,三分钟
2,3,This is it . Ground zero .,时间到，准备引爆
3,4,Want to say a few words for the occasion ?,要说点什么来纪念这一刻吗？
4,5,"With a gun barrel between your teeth , you spe...",当你嘴里含着枪管只能支支吾吾地说话
...,...,...,...
2065,2066,- You shot yourself ?,-你开枪射你自己？
2066,2067,"- Yes , but it 's OK . Marla , look at me .",-对，没事了，玛拉，看着我
2067,2068,I 'm really OK .,我真的没事
2068,2069,Trust me . Everything 's gonna be fine .,相信我一切都没事了


In [124]:
convo_df_zh_en.to_csv('temp.csv')